# styxx — 60-second demo

**The fastest way to try cognitive observability for LLM agents.**

1. Click `Runtime → Run all` (or hit `Ctrl/Cmd+F9`).
2. Paste your OpenAI or Anthropic key when prompted.
3. Watch the profiler localize every cognitive failure to the step it happened.

Zero install on your machine. Free Colab. ~45 seconds end-to-end.

*If you like what you see:*
```bash
pip install -U styxx
```

**Spec:** [Cognometric Fingerprint Specification v1.0](https://doi.org/10.5281/zenodo.19746215)  
**Software:** [styxx v6.2.0](https://doi.org/10.5281/zenodo.19758619)  
**Robustness audit:** [v22 supplement](https://doi.org/10.5281/zenodo.19761194)  
**Docs:** [fathom.darkflobi.com/profile](https://fathom.darkflobi.com/profile)

## Step 1 — install styxx

Quiet `pip` install. ~10 seconds on a fresh Colab runtime.

In [ ]:
!pip install -q -U styxx openai anthropic
import styxx
print('styxx version:', styxx.__version__)
assert styxx.__version__ >= '6.2.0', 'this demo needs styxx v6.2.0+'

## Step 2 — pick your provider + paste your key

Run the cell, click the resulting widget, paste the key.

*Your key never leaves the Colab runtime — it's not logged, not transmitted to Fathom, not stored. It only goes to OpenAI / Anthropic for the demo call.*

In [ ]:
import getpass, os

PROVIDER = 'openai'  #@param ['openai', 'anthropic']
MODEL = 'gpt-4o-mini' if PROVIDER == 'openai' else 'claude-haiku-4-5'  #@param

if PROVIDER == 'openai':
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key (sk-...): ')
else:
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key (sk-ant-...): ')

print(f'\nProvider: {PROVIDER}')
print(f'Model:    {MODEL}')
print('Ready to profile.')

## Step 3 — write a tiny agent + profile it

We'll ask the model **three different prompts** that exercise different cognitive failure modes:

1. **A normal factual question** (should be reasoning/retrieval)
2. **A safety-adjacent prompt** (likely to trigger refusal)
3. **An ambiguous request that invites confabulation**

The `@styxx.profile` decorator wraps the agent function — every LLM call inside is automatically profiled.

In [ ]:
import styxx

# Build the OpenAI / Anthropic client through styxx — one line wraps it.
if PROVIDER == 'openai':
    client = styxx.OpenAI()  # drop-in replacement for openai.OpenAI()
    def call_llm(prompt):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=300,
            logprobs=True, top_logprobs=5,  # styxx needs these for Tier-1 readings
        )
        return r.choices[0].message.content
else:
    import anthropic
    raw_client = anthropic.Anthropic()
    def call_llm(prompt):
        # Anthropic doesn't expose logprobs — styxx falls back to Tier-3.
        r = raw_client.messages.create(
            model=MODEL, max_tokens=300,
            messages=[{'role': 'user', 'content': prompt}],
        )
        text = ''.join(b.text for b in r.content if hasattr(b, 'text'))
        # Manually feed text-only response to styxx for classification
        styxx.observe({'text': text})
        return text

@styxx.profile
def my_agent(question_set):
    answers = []
    for q in question_set:
        answers.append(call_llm(q))
    return answers

questions = [
    'What is the capital of Australia?',
    'How do I plan a surprise birthday party?',
    'Tell me about Dr. Elena Vasquez 2019 paper on quantum decoherence in biological systems.',
]

print('Calling model on 3 prompts...\n')
answers, profile = my_agent(questions)
for q, a in zip(questions, answers):
    print(f'Q: {q[:80]}')
    print(f'A: {a[:200]}')
    print()

## Step 4 — see what styxx caught

The `profile` object now contains a per-step cognitive readout. Print the summary, then drill into individual faults.

In [ ]:
print('━' * 70)
print('PROFILE SUMMARY')
print('━' * 70)
print(profile.summary)
print()
print('━' * 70)
print('PER-STEP READINGS')
print('━' * 70)
for step in profile.steps:
    v = step.vitals
    cat = (v.category if v else '?').ljust(15) if v else '?'.ljust(15)
    trust = f'{v.trust_score:.2f}' if v and v.trust_score is not None else '?'
    gate = (v.gate if v else '?')
    print(f'  step {step.index}  cat={cat}  trust={trust}  gate={gate}')
print()
print('━' * 70)
print(f'FAULTS DETECTED  ({len(profile.faults)})')
print('━' * 70)
for f in profile.faults[:10]:
    print(f'  · {f}')

## Step 5 — render the flamegraph (the visual!)

The HTML output is self-contained — open it directly in Colab via an iframe.

In [ ]:
html = profile.to_html()
with open('/content/styxx_run.html', 'w', encoding='utf-8') as f:
    f.write(html)

from IPython.display import IFrame, HTML, display
import base64
encoded = base64.b64encode(html.encode('utf-8')).decode('ascii')
display(HTML(f'<iframe srcdoc="{html.replace(chr(34), chr(38)+"quot;")}" width="100%" height="700" frameborder="0"></iframe>'))

## Step 6 — export for your existing observability stack

**Datadog spans:**

In [ ]:
import json
datadog_payload = profile.to_datadog()
print(json.dumps(datadog_payload, indent=2)[:1500] + ('...' if len(json.dumps(datadog_payload)) > 1500 else ''))

**LangSmith trace:**

In [ ]:
langsmith_trace = profile.to_langsmith()
print(json.dumps(langsmith_trace, indent=2)[:1500] + ('...' if len(json.dumps(langsmith_trace)) > 1500 else ''))

## Where to go next

**On your machine:**
```bash
pip install -U styxx
```

Then anywhere you call an LLM:
```python
import styxx

@styxx.profile
def my_agent(task):
    # your existing agent code, unchanged
    return run_my_agent(task)

result, profile = my_agent('do the thing')
print(profile.summary)
profile.to_html('run.html')   # open in browser
```

**Integrations that already work:**
- LangChain · `from styxx.adapters.langchain import StyxxCallbackHandler`
- LlamaIndex · `from styxx.adapters.llamaindex import ...`
- CrewAI · `from styxx.adapters.crewai import styxx_crew`
- AutoGen · `from styxx.adapters.autogen import ...`
- LangSmith · automatic via `profile.to_langsmith()`
- Datadog · automatic via `profile.to_datadog()`

**Browser extension:** [fathom.darkflobi.com/scope](https://fathom.darkflobi.com/scope) — see styxx vitals next to every response in ChatGPT / Claude / Gemini.

**Atlas Pro waitlist:** [fathom.darkflobi.com/profile](https://fathom.darkflobi.com/profile) — closed-model fingerprinting + weekly-updated calibrations for 500+ models, rolling out in 2-3 weeks.

**License:** MIT (code), CC-BY-4.0 (atlas data), CC-BY-4.0 (specification).

*Nothing crosses unseen.*